# MI-projet


## Installation de l'environnement :

In [1]:
! python -m pip install numpy fairlearn plotly nbformat ipykernel aif360["inFairness"] aif360['AdversarialDebiasing'] causal-learn BlackBoxAuditing cvxpy dice-ml lime shapkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 39.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 22.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.8/191.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.5/268.5 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Imports

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
from aif360.sklearn.metrics import *

/usr/local/lib/python3.12/dist-packages/inFairness/utils/ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
/usr/local/lib/python3.12/dist-packages/inFairness/utils/ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  monte_carlo_vect_ndcg = vmap(vect_normalized_discounted

### Chargement du Dataset

In [3]:
df = pd.read_csv("Berrahma_Mohamed_Amine.csv")

# Introduction :
L’intelligence artificielle joue un rôle important dans le domaine médical, notamment dans l’analyse d’images radiologiques. Les modèles de deep learning appliqués aux radiographies thoraciques peuvent assister les médecins dans le diagnostic de diverses maladies. Toutefois, ces systèmes peuvent reproduire ou amplifier des biais présents dans les données, ce qui peut conduire à des performances inéquitables selon certains groupes de patients.

Le cas d’usage étudié dans ce projet repose sur un sous-ensemble du dataset NIH Chest X-ray, composé de métadonnées associées à des radiographies thoraciques. Ce dataset contient des informations diverses (âge, genre...etc) ainsi que des annotations médicales indiquant la présence ou l’absence de maladies.

L’objectif principal de ce projet est d’identifier et d’analyser les biais potentiels présents dans ces données, en particulier en lien avec des attributs sensibles tels que le genre et l’âge. Pour cela, dans un premier temps une analyse descriptive est réalisée afin d’examiner la distribution des maladies selon ces groupes.

Dans un second temps, des métriques d’équité telles que la Statistical Parity Difference (SPD) et le Disparate Impact (DI) sont utilisées pour quantifier les déséquilibres observés.
Enfin, une méthode de mitigation par preprocessing, basée sur l’algorithme Reweighing, est appliquée afin de réduire ces biais et produire un dataset transformé plus équitable.

# 1 - Préparation des données :

Dans cette partie, une analyse est réalisée afin d’observer les dimensions du dataset, les types de variables ainsi que leur distribution. Une attention particulière est portée à la détection de valeurs abhérrantes, notamment sur la variable âge, afin de garantir la cohérence des informations.

Enfin, certaines transformations sont effectuées pour faciliter l’analyse, notamment la création d’une variable binaire indiquant la présence ou l’absence de maladie, ainsi que la catégorisation de l’âge en groupes. Ces étapes permettent d’obtenir un dataset propre, structuré et adapté à l’analyse des biais

# Informations sur le Dataset

In [4]:
df.shape
df.info()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54252 entries, 0 to 54251
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Image Index                  54252 non-null  object 
 1   Finding Labels               54252 non-null  object 
 2   Follow-up #                  54252 non-null  int64  
 3   Patient ID                   54252 non-null  int64  
 4   Patient Age                  54252 non-null  int64  
 5   Patient Gender               54252 non-null  object 
 6   View Position                54252 non-null  object 
 7   OriginalImage[Width          54252 non-null  int64  
 8   Height]                      54252 non-null  int64  
 9   OriginalImagePixelSpacing[x  54252 non-null  float64
 10  y]                           54252 non-null  float64
dtypes: float64(2), int64(5), object(4)
memory usage: 4.6+ MB


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
0,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171000,0.171000
1,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143000,0.143000
2,00000003_001.png,Hernia,1,3,74,F,PA,2500,2048,0.168000,0.168000
3,00000003_002.png,Hernia,2,3,75,F,PA,2048,2500,0.168000,0.168000
4,00000003_003.png,Hernia|Infiltration,3,3,76,F,PA,2698,2991,0.143000,0.143000
...,...,...,...,...,...,...,...,...,...,...,...
54247,00030793_000.png,Mass|Nodule,0,30793,58,F,PA,2021,2021,0.194311,0.194311
54248,00030794_000.png,No Finding,0,30794,38,F,PA,2021,2021,0.194311,0.194311
54249,00030796_000.png,No Finding,0,30796,44,M,PA,2021,2021,0.194311,0.194311
54250,00030798_000.png,No Finding,0,30798,30,M,PA,2500,2048,0.171000,0.171000


In [5]:
df.isna().sum()

,0
Image Index,0
Finding Labels,0
Follow-up #,0
Patient ID,0
Patient Age,0
Patient Gender,0
View Position,0
OriginalImage[Width,0
Height],0
OriginalImagePixelSpacing[x,0


Ici on voit qu'il n'ya aucun Nan toutes nos cases ont des nombres, il n'y a pas de cellules vides ou manquantes. On confirme cela grace a la cellule df.isna.sum qui vaut 0 partout.

In [6]:
df.describe()

,Follow-up #,Patient ID,Patient Age,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
count,54252.000000,54252.000000,54252.000000,54252.000000,54252.000000,54252.000000,54252.000000
mean,8.377829,14368.131663,46.702702,2648.508516,2488.283455,0.155542,0.155542
std,14.537982,8406.053815,17.071171,341.743128,400.517916,0.016171,0.016171
min,0.000000,2.000000,1.000000,1215.000000,966.000000,0.115000,0.115000
25%,0.000000,7262.750000,34.000000,2500.000000,2048.000000,0.143000,0.143000
50%,3.000000,14120.500000,49.000000,2530.000000,2544.000000,0.143000,0.143000
75%,9.000000,20680.000000,59.000000,2992.000000,2991.000000,0.168000,0.168000
max,156.000000,30802.000000,413.000000,3305.000000,3056.000000,0.198800,0.198800


In [7]:
df.sort_values("Patient Age", ascending=False).head(10)

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
41802,00021275_003.png,No Finding,3,21275,413,F,AP,3056,2544,0.139,0.139
41318,00021047_002.png,Mass|Pleural_Thickening,2,21047,412,M,AP,3056,2544,0.139,0.139
10219,00005567_000.png,Effusion|Pneumonia,0,5567,412,M,AP,3056,2544,0.139,0.139
41008,00020900_002.png,No Finding,2,20900,411,M,AP,3056,2544,0.139,0.139
47820,00026028_001.png,Atelectasis,1,26028,154,M,PA,2992,2991,0.143,0.143
46498,00025206_000.png,Infiltration|Mass,0,25206,153,M,PA,2992,2991,0.143,0.143
36029,00018366_044.png,Pneumothorax,44,18366,152,F,PA,2302,2991,0.143,0.143
30018,00015558_000.png,No Finding,0,15558,149,M,PA,2992,2991,0.143,0.143
23189,00012238_010.png,No Finding,10,12238,148,M,PA,2992,2991,0.143,0.143
27807,00014456_000.png,No Finding,0,14456,95,F,PA,2522,2991,0.143,0.143


In [8]:
len(df["Finding Labels"].unique())

632

Grace aux 3 cellules précédentes : on voit que l'age maximal est de 413 ce qui n'est pas cohérent, c'est une valeur abhérrante. De plus on a 632 labels différents ce qui est enorme donc l'idée sera de transformer cette colonne en ou un label binaire:finding\no finding.

### Gestion des données abhérrantes

L’analyse descriptive de la variable Patient Age a mis en évidence la présence de valeurs abhérrantes, notamment des âges supérieurs à 120 ans, biologiquement improbables.

Plutôt que de supprimer directement les lignes dans le dataframe, une vérification a été effectuée au niveau de chaque Patient ID afin d’examiner si d’autres enregistrements du même patient comportaient un âge cohérent.

Lorsque c’était le cas, l’âge abhérrant a été corrigé en utilisant l’âge de référence du patient, défini comme l'âge le plus fréquent ou sinon la médiane des âges observés.

In [9]:
df.duplicated().sum() #On verifie qu'il ya pas de doublons
df["Patient Gender"].unique() # On verifie que pour le genre il y'a M et F

array(['M', 'F'], dtype=object)

In [11]:
df["Patient Age BEFORE"] = df["Patient Age"]# je garde une trace pour comparer plus tard

In [12]:
def ref_age_func(s):
    s = s[(s >= 1) & (s <= 120)]
    m = s.mode()
    return m.iloc[0] if len(m) else s.median()

ref_age = df.groupby("Patient ID")["Patient Age"].apply(ref_age_func)

df.loc[df["Patient Age"] > 120, "Patient Age"] = df["Patient ID"].map(ref_age)

Visualisation du résultat


In [13]:
df.loc[df["Patient Age BEFORE"] > 120, ["Patient ID", "Patient Age BEFORE", "Patient Age"]].head(20)



,Patient ID,Patient Age BEFORE,Patient Age
10219,5567,412,53
23189,12238,148,63
30018,15558,149,46
36029,18366,152,64
41008,20900,411,71
41318,21047,412,52
41802,21275,413,19
46498,25206,153,36
47820,26028,154,60


On voit bien ici que les ages ont été corrigés passant par exemple de 412 a 53 ans...

In [14]:
import plotly.express as px

fig = px.histogram(df, x=["Patient Age BEFORE", "Patient Age"],
                   barmode="overlay",
                   title="Distribution de l'âge avant et après nettoyage")

fig.show()

On voit l'avant/apres en histogramme, en effet il y'a des petits points bleu vers les 150 et les 400 ans, mais c'est compliqué à voir carb c'est tres zoomé en effet il y'a tres peu de lignes dans le dataframe avec des ages abhérrants.

In [15]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_box(y=df["Patient Age BEFORE"], name="Avant nettoyage")
fig.add_box(y=df["Patient Age"], name="Après nettoyage")

fig.update_layout(title="Boxplot de l'âge avant et après nettoyage")

fig.show()


On visualise un peu mieux avec ce graphique, les petits points bleus à environ 400 et 150 ans ne sont plus la après le nettoyage

In [16]:
df = df.drop(columns=["Patient Age BEFORE"])# On n'a plus besoin de cette colonne donc on l'enlève pour épurer notre dataframe

###**Trasformation des colonnes :**

In [17]:
# Créer la variable binaire
df["has_disease"] = (df["Finding Labels"] != "No Finding").astype(int)


La variable Finding Labels contient plusieurs catégories médicales, elle a été conservée afin de préserver l’information originale. Une variable binaire has_disease, indiquant la présence (1) ou l’absence (0) de maladie a été ajoutée.
Cet ajout permet de comparer la probabilité de maladie entre différents groupes, notamment selon le genre et l’âge.

In [18]:
bins = [0, 20, 40, 60, 120]
labels = ["<20", "20-40", "40-60", "60+"]

df["age_group"] = pd.cut(df["Patient Age"], bins=bins, labels=labels)
df["age_group"].value_counts()
df

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],has_disease,age_group
0,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171000,0.171000,0,60+
1,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143000,0.143000,1,60+
2,00000003_001.png,Hernia,1,3,74,F,PA,2500,2048,0.168000,0.168000,1,60+
3,00000003_002.png,Hernia,2,3,75,F,PA,2048,2500,0.168000,0.168000,1,60+
4,00000003_003.png,Hernia|Infiltration,3,3,76,F,PA,2698,2991,0.143000,0.143000,1,60+
...,...,...,...,...,...,...,...,...,...,...,...,...,...
54247,00030793_000.png,Mass|Nodule,0,30793,58,F,PA,2021,2021,0.194311,0.194311,1,40-60
54248,00030794_000.png,No Finding,0,30794,38,F,PA,2021,2021,0.194311,0.194311,0,20-40
54249,00030796_000.png,No Finding,0,30796,44,M,PA,2021,2021,0.194311,0.194311,0,40-60
54250,00030798_000.png,No Finding,0,30798,30,M,PA,2500,2048,0.171000,0.171000,0,20-40


La variable continue Patient Age a été conservée afin de préserver l’information originale. Une nouvelle variable catégorielle age_group a été créée, regroupant les patients en tranches d’âge (<20, 20–40, 40–60, 60+). Cette approche permet de conserver la précision de l’âge tout en facilitant l’analyse comparative entre groupes. La variable age_group est utile pour l’étude des biais, car elle permet d’identifier d’éventuelles disparités entre différentes catégories d’âge de manière plus interprétable qu’une variable continue.

## Conclusion de la partie :
La préparation des données a permis de vérifier la qualité et la cohérence du dataset, l’analyse initiale a confirmé l’absence de valeurs manquantes et de doublons, tandis que la détection et la correction des âges aberrants ont permis d’assurer la fiabilité des informations.

La transformation de certaines variables, notamment la création d’une variable binaire de présence de maladie et la catégorisation de l’âge en groupes, a permis d’adapter les données à l’analyse des biais.

Le dataset est désormais propre, structuré et prêt pour l’analyse descriptive et l’étude des éventuelles disparités entre groupes démographiques.

##**2**- Analyse descriptive et observation des biais

Dans cette partie, une analyse univariée est d’abord réalisée afin de comprendre la distribution individuelle des variables clés, telles que le genre, l’âge et le statut de maladie.

Une analyse bivariée permet ensuite d’examiner la relation entre les attributs sensibles (genre et groupes d’âge) et le label, afin de détecter d’éventuelles disparités.

Enfin, une analyse multivariée approfondit l’étude en combinant plusieurs variables, suivie du calcul de métriques d’équité telles que le Statistical Parity Difference et le Disparate Impact, ce qui permet de quantifier les biais potentiels présents dans les données.

### A) Analyse univariée

#### -Distribution globale de la maladie

In [19]:
df["disease_status"] = df["has_disease"].map({
    0: "no_disease",
    1: "has_disease"
})
#Version textuelle de la colonne has_disease
df

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],has_disease,age_group,disease_status
0,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171000,0.171000,0,60+,no_disease
1,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143000,0.143000,1,60+,has_disease
2,00000003_001.png,Hernia,1,3,74,F,PA,2500,2048,0.168000,0.168000,1,60+,has_disease
3,00000003_002.png,Hernia,2,3,75,F,PA,2048,2500,0.168000,0.168000,1,60+,has_disease
4,00000003_003.png,Hernia|Infiltration,3,3,76,F,PA,2698,2991,0.143000,0.143000,1,60+,has_disease
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54247,00030793_000.png,Mass|Nodule,0,30793,58,F,PA,2021,2021,0.194311,0.194311,1,40-60,has_disease
54248,00030794_000.png,No Finding,0,30794,38,F,PA,2021,2021,0.194311,0.194311,0,20-40,no_disease
54249,00030796_000.png,No Finding,0,30796,44,M,PA,2021,2021,0.194311,0.194311,0,40-60,no_disease
54250,00030798_000.png,No Finding,0,30798,30,M,PA,2500,2048,0.171000,0.171000,0,20-40,no_disease


In [20]:
df["disease_status"].value_counts(normalize=True)


,proportion
disease_status,
no_disease,0.538432
has_disease,0.461568


In [21]:
import plotly.express as px

fig = px.bar(
    df["disease_status"].value_counts(),
    title="Distribution globale de la maladie",
    labels={"value": "Nombre de patients", "index": "Statut"}
)
fig.show()

**Interpretation** :

L’analyse univariée du statut de maladie montre qu'environ 54 % des observations ne présentent aucune pathologie, contre 46 % présentant au moins une maladie.
Cette distribution est relativement équilibrée.

#### -Distribution des groupes d'âges:


In [22]:
df["age_group"].value_counts(normalize=True)

,proportion
age_group,
40-60,0.439007
20-40,0.267124
60+,0.219347
<20,0.074523


In [23]:
fig = px.bar(
    df["age_group"].value_counts().sort_index(),
    title="Distribution des groupes d'âge",
    labels={"value": "Nombre de patients", "index": "Groupe d'âge"}
)

fig.show()

**Interpretation:**
L’analyse univariée des groupes d’âge montre que la majorité des observations concernent des patients âgés de 40 à 60 ans (43 %). Les patients de moins de 20 ans sont fortement sous-représentés (7 %), tandis que les groupes 20–40 ans et 60+ représentent respectivement 26 % et 21 % du dataset.

Cette distribution reflète une prédominance des adultes dans les données et pourrait influencer l’analyse des biais, notamment pour les groupes les moins représentés commes les jeunes de <20 ans

#### -Distribution du genre:

In [24]:
df["Patient Gender"].value_counts(normalize=True)


,proportion
Patient Gender,
M,0.567131
F,0.432869


In [25]:
fig = px.bar(
    df["Patient Gender"].value_counts().sort_index(),
    title="Distribution du genre",
    labels={"value": "Nombre de patients", "index": "Genre"}
)

fig.show()

**Interpretation**: L'analyse univariée sur le genre montre une légère prédominance des hommes sur les femmes, representés respectivement par 57% et 43% des observations.
Cela ne constitue pas un désequilibre majeur mais il faut le prendre en compte lors de l'analyse de biais.

### B) Analyse bivariée

#### **Analyse des biais selon le genre**:

In [26]:
#Tableau croisé : comme dans le TD

gender_biais = pd.crosstab(
    df["Patient Gender"],
    df["has_disease"],
    normalize="index"
)

gender_biais

has_disease,0,1
Patient Gender,,
F,0.538963,0.461037
M,0.538027,0.461973


In [27]:
#Calcul explicit des probabilités :

p_male = df[df["Patient Gender"]=="M"]["has_disease"].mean()
p_female = df[df["Patient Gender"]=="F"]["has_disease"].mean()

print("P(maladie | homme) =", p_male)
print("P(maladie | femme) =", p_female)

P(maladie | homme) = 0.4619734789391576
P(maladie | femme) = 0.4610373019928462


In [28]:
plot_prop = (
    df.groupby(["Patient Gender", "disease_status"])
      .size()
      .reset_index(name="count")
)

# Calcul des proportions manuellement
plot_prop["proportion"] = (
    plot_prop["count"] /
    plot_prop.groupby("Patient Gender")["count"].transform("sum")
)

fig = px.bar(
    plot_prop,
    x="Patient Gender",
    y="proportion",
    color="disease_status",
    barmode="group",
    title="Proportion de maladie selon le genre",
    color_discrete_map={
        "no_disease": "#1f77b4",
        "has_disease": "#d62728"
    }
)

fig.update_layout(yaxis_title="Proportion")

fig.show()


**Interpetation:** L’analyse bivariée selon le genre montre que la probabilité d’avoir une maladie est quasiment identique entre hommes (46.20 %) et femmes (46.10 %). Il semble y avoir une absence de biais.

#### Analyse des biais selon l’âge:

In [29]:
#Tableau croisé :

age_biais = pd.crosstab(
    df["age_group"],
    df["has_disease"],
    normalize="index"
)

age_biais

has_disease,0,1
age_group,,
<20,0.587435,0.412565
20-40,0.575214,0.424786
40-60,0.534744,0.465256
60+,0.484370,0.515630


In [30]:
plot_age = (
    df.groupby("age_group")["has_disease"]
      .mean()
      .reset_index()
)

fig = px.bar(
    plot_age,
    x="age_group",
    y="has_disease",
    title="Probabilité de maladie selon le groupe d'âge",
)

fig.update_layout(yaxis_title="Probabilité de maladie")

fig.show()


/tmp/ipython-input-652672950.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



**Interpretation:**   L’analyse bivariée selon les groupes d’âge montre une augmentation progressive de la probabilité de maladie avec l’âge. Les patients de moins de 20 ans présentent une probabilité de maladie de 41 %, tandis que ce taux atteint 51.6 % chez les patients de 60 ans et plus.

La différence entre ces 2 groupes extremes est notable.

Mais meme si cette différence est justifiable biologiquement (les personnes agées ont plus tendance a tomber malade), elle représente néanmoins un déséquilibre dans le dataset, justifiant l’analyse et la mitigation des biais.

### C) Analyse croisée (multivariée)
**Age et genre**

In [31]:
biais_croise = pd.crosstab(
    [df["age_group"], df["Patient Gender"]],
    df["has_disease"],
    normalize="index"
)

biais_croise


has_disease                      0         1
age_group Patient Gender                    
<20       F               0.614173  0.385827
          M               0.568980  0.431020
20-40     F               0.592112  0.407888
          M               0.562171  0.437829
40-60     F               0.526369  0.473631
          M               0.541815  0.458185
60+       F               0.469136  0.530864
          M               0.494027  0.505973

In [32]:
fig = px.histogram(
    df,
    x="age_group",
    color="disease_status",
    facet_col="Patient Gender",
    barmode="group",
    title="Distribution de la maladie selon l'âge et le genre"
)

fig.show()

In [34]:
heatmap_df = (
    df.groupby(["age_group", "Patient Gender"])["has_disease"]
    .mean()
    .reset_index()
)

fig = px.density_heatmap(
    heatmap_df,
    x="age_group",
    y="Patient Gender",
    z="has_disease",
    color_continuous_scale="RdBu_r",
    title="Probabilité de maladie selon l'âge et le genre"
)

fig.show()


/tmp/ipython-input-1730769329.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



**Interpretation :** L’analyse multivariée combinant âge et genre montre que la probabilité de maladie augmente avec l’âge pour les deux genres.

Les différences entre hommes et femmes restent modérées et varient selon les tranches d’âge. Chez les jeunes (<20 et 20–40 ans), les hommes présentent une probabilité légèrement plus élevée de maladie. En revanche, chez les groupes plus âgés (40–60 et 60+), les femmes présentent une probabilité légèrement supérieure.

Ces écarts restent toutefois faibles comparés à l’effet global de l’âge, indiquant que l’âge constitue le principal facteur de déséquilibre dans le dataset, tandis que le biais lié au genre demeure limité.

### D) Calcul des métriques fairness

### Biais sur le genre: 

In [35]:
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric

df_aif = df[["has_disease", "Patient Gender"]].copy()
df_aif["gender_bin"] = df_aif["Patient Gender"].map({"M": 1, "F": 0})

dataset = BinaryLabelDataset(
    df=df_aif[["has_disease", "gender_bin"]],
    label_names=["has_disease"],
    protected_attribute_names=["gender_bin"]
)

#Convention du TD
priv = [{"gender_bin": 1}]      # M
unpriv = [{"gender_bin": 0}]    # F


SPD/DI global:

In [36]:
metric = BinaryLabelDatasetMetric(
    dataset,
    privileged_groups=priv,
    unprivileged_groups=unpriv
)

print("SPD (AIF360):", metric.statistical_parity_difference())
print("DI  (AIF360):", metric.disparate_impact())

SPD (AIF360): -0.0009361769463113734
DI  (AIF360): 0.9979735266438646


**Interpretation :** le SPD est proche de 0 et le DI est proche de 1 donc il n'existe pas de biais global lié au genre

SPD/DI par groupe d'age:

In [37]:
results_aif = []

for age in sorted(df["age_group"].dropna().unique()):
    sub = df[df["age_group"] == age].copy()

    sub_aif = sub[["has_disease", "Patient Gender"]].copy()
    sub_aif["gender_bin"] = sub_aif["Patient Gender"].map({"M": 1, "F": 0})

    dataset_sub = BinaryLabelDataset(
        df=sub_aif[["has_disease", "gender_bin"]],
        label_names=["has_disease"],
        protected_attribute_names=["gender_bin"]
    )

    m = BinaryLabelDatasetMetric(
        dataset_sub,
        privileged_groups=priv,
        unprivileged_groups=unpriv
    )

    results_aif.append({
        "age_group": str(age),
        "SPD_AIF360": m.statistical_parity_difference(),
        "DI_AIF360": m.disparate_impact(),
        "n": len(sub)
    })

fairness_age_aif360 = pd.DataFrame(results_aif)
fairness_age_aif360


,age_group,SPD_AIF360,DI_AIF360,n
0,20-40,-0.029940,0.931617,14492
1,40-60,0.015446,1.033712,23817
2,60+,0.024891,1.049195,11900
3,<20,-0.045193,0.895148,4043


**Interpretation :** C'est assez equitable mais il y'a un leger biais dans la tranche d'age des - de 20 ans mais le groupe ne contient pas beaucoup de patients donc pas vraiment representatif.

Calcul des metriques à la main :

In [38]:
#Statistical Parity Difference :
spd = p_female - p_male
print("Statistical Parity Difference:", spd)


Statistical Parity Difference: -0.0009361769463113734


In [39]:
#Disparate Impact :
di = p_female / p_male
print("Disparate Impact:", di)

Disparate Impact: 0.9979735266438646


**Interpretation finale :** Les métriques fairness montrent l’absence de biais significatif lié au genre au niveau global, avec un Statistical Parity Difference proche de zéro et un Disparate Impact proche de 1.

Une analyse par groupe d’âge révèle de légers écarts dans certaines tranches, notamment chez les patients de moins de 20 ans, où le biais est plus marqué.

Ainsi, le dataset peut être considéré comme globalement équitable vis-à-vis du genre, malgré quelques variations locales.

####**biais sur l'age :**

In [40]:
df_aif_age = df[["has_disease", "age_group"]].copy()

df_aif_age["age_bin"] = df_aif_age["age_group"].map({
    "<20": 0,
    "20-40": 0,
    "40-60": 0,
    "60+": 1
}) # par convention le groupe privilegié sera celui de 60+

dataset_age = BinaryLabelDataset(
    df=df_aif_age[["has_disease", "age_bin"]],
    label_names=["has_disease"],
    protected_attribute_names=["age_bin"]
)
priv_age = [{"age_bin": 1}]      # 60+
unpriv_age = [{"age_bin": 0}]    # <60



In [41]:
metric_age = BinaryLabelDatasetMetric(
    dataset_age,
    privileged_groups=priv_age,
    unprivileged_groups=unpriv_age
)

print("SPD (Age):", metric_age.statistical_parity_difference())
print("DI  (Age):", metric_age.disparate_impact())


SPD (Age): -0.06925227703472775
DI  (Age): 0.8656939216569002


**Interpretation :**
L’analyse fairness selon l’âge révèle un Statistical Parity Difference de -0.069 et un Disparate Impact de 0.866. Il y'a presence d'un biais modéré, les patients de 60 ans et plus présentent une probabilité significativement plus élevée d’avoir une maladie que les patients plus jeunes.

Bien que ce déséquilibre puisse refléter une réalité médicale (augmentation des pathologies avec l’âge), il constitue néanmoins un écart statistique dans le dataset, justifiant l’application d’une méthode de mitigation afin d’améliorer l’équité globale des données.

##**Conclusion de l'analyse descriptive :**
L’analyse descriptive a montré que le dataset est globalement équilibré en termes de présence de maladie (53 % sans maladie contre 47 % avec maladie). L’étude selon le genre révèle des probabilités de maladie presque identiques entre hommes et femmes, confirmées par des métriques fairness proches des valeurs idéales (spd proche de 0 ; DI proche de 1), indiquant l’absence de biais significatif lié au genre.

En revanche, l’analyse selon l’âge met en évidence une augmentation de la probabilité de maladie avec l’âge, conduisant à un déséquilibre dans le dataset entre les patients de 60 ans et plus et les patients plus jeunes (SPD = -0.069 ; DI = 0.866).

Ainsi, si le dataset apparaît équitable vis-à-vis du genre, un biais mesurable lié à l’âge justifie l’application d’une méthode de mitigation.

##**3**- Méthode de mitigation des biais par pré-processing

Suite à l’analyse descriptive, un déséquilibre statistique a été identifié, notamment en lien avec l’âge. Afin de réduire ces disparités, une méthode de mitigation par preprocessing appelée data reweighting est appliquée sur les attributs sensibles que sont le genre et l’âge.

La méthode de data reweighting consiste à attribuer un poids spécifique à chaque observation en fonction de son appartenance à un groupe et de son statut de maladie. L’objectif est de rééquilibrer statistiquement les groupes privilégiés et non privilégiés sans supprimer de données, en ajustant leur contribution lors de l’apprentissage d’un modèle.

Cette approche permet ainsi d’améliorer l’équité du dataset tout en conservant l’ensemble des informations disponibles.

###**Mitigation sur le genre :**

In [42]:
# garder seulement les colonnes utiles
df_aif = df[["has_disease", "Patient Gender"]].copy()

# convertir gender en numérique
df_aif["Patient Gender"] = df_aif["Patient Gender"].map({"M": 1, "F": 0})

dataset = BinaryLabelDataset(
    df=df_aif,
    label_names=["has_disease"],
    protected_attribute_names=["Patient Gender"]
)


**Metrique avant :**

In [43]:
metric_before = BinaryLabelDatasetMetric(
    dataset,
    privileged_groups=[{"Patient Gender": 1}],
    unprivileged_groups=[{"Patient Gender": 0}]
)

print("SPD avant:", metric_before.statistical_parity_difference())
print("DI avant:", metric_before.disparate_impact())


SPD avant: -0.0009361769463113734
DI avant: 0.9979735266438646


**Data Reweighting:**

In [44]:
from aif360.algorithms.preprocessing import Reweighing

RW = Reweighing(
    privileged_groups=[{"Patient Gender": 1}],
    unprivileged_groups=[{"Patient Gender": 0}]
)

dataset_transformed = RW.fit_transform(dataset)

**Metrique après:**

In [45]:
metric_after = BinaryLabelDatasetMetric(
    dataset_transformed,
    privileged_groups=[{"Patient Gender": 1}],
    unprivileged_groups=[{"Patient Gender": 0}]
)

print("SPD après:", metric_after.statistical_parity_difference())
print("DI après:", metric_after.disparate_impact())

SPD après: 1.1102230246251565e-16
DI après: 1.0000000000000002


In [55]:
df["weight_genre"] = dataset_transformed.instance_weights
df.to_csv("berrahma_dataset_mitigated_aif360.csv", index=False)


In [57]:
comparaison = {
    "Metric": ["SPD", "DI"],
    "Before": [
        metric_before.statistical_parity_difference(),
        metric_before.disparate_impact()
    ],
    "After": [
        metric_after.statistical_parity_difference(),
        metric_after.disparate_impact()
    ]
}

import pandas as pd
pd.DataFrame(comparaison)


,Metric,Before,After
0,SPD,-0.000936,1.110223e-16
1,DI,0.997974,1.000000e+00


On voit qu'après application de la méthode de reweighing les metriques SPD et DI se rapprochent de leur valeurs ideales 0 et 1 ce qui indique une amélioration de l'équité entre les groupes.
L'amélioration est légère puisque pour le genre c'etait déjà plutot équilibré.

###**Mitigation sur l'age :**

In [58]:
df["age_bin"] = df["age_group"].map({
    "<20": 0,
    "20-40": 0,
    "40-60": 0,
    "60+": 1
})

df_aif_age = df[["has_disease", "age_bin"]].copy()

dataset_age = BinaryLabelDataset(
    df=df_aif_age,
    label_names=["has_disease"],
    protected_attribute_names=["age_bin"]
)

# Convention TD
priv_age = [{"age_bin": 1}]
unpriv_age = [{"age_bin": 0}]

**Metrique avant :**

In [59]:
metric_age_before = BinaryLabelDatasetMetric(
    dataset_age,
    privileged_groups=priv_age,
    unprivileged_groups=unpriv_age
)

print("SPD avant mitigation (Age):", metric_age_before.statistical_parity_difference())
print("DI avant mitigation (Age):", metric_age_before.disparate_impact())

SPD avant mitigation (Age): -0.06925227703472775
DI avant mitigation (Age): 0.8656939216569002


**Data Reweighting:**

In [60]:
RW_age = Reweighing(
    privileged_groups=priv_age,
    unprivileged_groups=unpriv_age
)

dataset_age_transformed = RW_age.fit_transform(dataset_age)

**Metrique après :**

In [61]:
metric_age_after = BinaryLabelDatasetMetric(
    dataset_age_transformed,
    privileged_groups=priv_age,
    unprivileged_groups=unpriv_age
)

print("SPD après mitigation (Age):", metric_age_after.statistical_parity_difference())
print("DI après mitigation (Age):", metric_age_after.disparate_impact())

SPD après mitigation (Age): -1.1102230246251565e-16
DI après mitigation (Age): 0.9999999999999998


In [62]:
df["weight_age"] = dataset_age_transformed.instance_weights
comparaison_age = pd.DataFrame({
    "Metric": ["SPD", "DI"],
    "Before": [
        metric_age_before.statistical_parity_difference(),
        metric_age_before.disparate_impact()
    ],
    "After": [
        metric_age_after.statistical_parity_difference(),
        metric_age_after.disparate_impact()
    ]
})

comparaison_age

,Metric,Before,After
0,SPD,-0.069252,-1.110223e-16
1,DI,0.865694,1.000000e+00


**Interpretation:** Pour l'age : le biais etait déjà plus important et l'amélioration est donc beaucoup plus importante ici, les SPD et DI se rapprochent de leurs valeurs ideales.

In [63]:
df.to_csv("dataset_mitigated_age_aif360.csv", index=False)
df

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],has_disease,age_group,disease_status,weight,age_bin,weight_age,weight_genre
0,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171000,0.171000,0,60+,no_disease,1.000753,1,1.111613,1.000753
1,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143000,0.143000,1,60+,has_disease,1.001152,1,0.895154,1.001152
2,00000003_001.png,Hernia,1,3,74,F,PA,2500,2048,0.168000,0.168000,1,60+,has_disease,1.001152,1,0.895154,1.001152
3,00000003_002.png,Hernia,2,3,75,F,PA,2048,2500,0.168000,0.168000,1,60+,has_disease,1.001152,1,0.895154,1.001152
4,00000003_003.png,Hernia|Infiltration,3,3,76,F,PA,2698,2991,0.143000,0.143000,1,60+,has_disease,1.001152,1,0.895154,1.001152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54247,00030793_000.png,Mass|Nodule,0,30793,58,F,PA,2021,2021,0.194311,0.194311,1,40-60,has_disease,1.001152,0,1.034030,1.001152
54248,00030794_000.png,No Finding,0,30794,38,F,PA,2021,2021,0.194311,0.194311,0,20-40,no_disease,0.999015,0,0.972562,0.999015
54249,00030796_000.png,No Finding,0,30796,44,M,PA,2021,2021,0.194311,0.194311,0,40-60,no_disease,1.000753,0,0.972562,1.000753
54250,00030798_000.png,No Finding,0,30798,30,M,PA,2500,2048,0.171000,0.171000,0,20-40,no_disease,1.000753,0,0.972562,1.000753


##**Conclusion de la partie :**
L’application de la méthode de data reweighting a permis de réduire les déséquilibres statistiques identifiés, en rapprochant les métriques fairness des valeurs idéales (SPD proche de 0 et DI proche de 1). L’amélioration est particulièrement visible pour l’attribut âge, où un biais modéré avait été observé.

Cette approche permet ainsi d’améliorer l’équité du dataset sans supprimer d’observations, en ajustant uniquement leur poids statistique.

##**Conclusion finale :**
Ce projet a permis d’identifier, d’analyser et de corriger les biais potentiels présents dans un sous-ensemble du dataset NIH Chest X-ray. Après une phase de préparation des données visant à garantir leur qualité et leur cohérence, une analyse descriptive approfondie a été menée afin d’examiner les disparités liées aux attributs sensibles que sont le genre et l’âge.

Les résultats ont montré une absence de biais significatif lié au genre, avec des probabilités de maladie quasi identiques entre hommes et femmes. En revanche, un déséquilibre statistique modéré a été observé selon l’âge, les patients de 60 ans et plus présentent une probabilité plus élevée de pathologie. Bien que cette différence puisse s’expliquer par des facteurs médicaux réels, elle constitue un écart mesurable au sens des métriques d’équité.

L’application de la méthode de mitigation par reweighing a permis de réduire ces déséquilibres en ajustant les poids des observations, améliorant ainsi les indicateurs fairness sans altérer les données originales.

Ce mini projet souligne l’importance d’auditer les datasets avant l’entraînement de modèles d’intelligence artificielle, particulièrement dans un contexte médical où l’équité et la fiabilité sont essentielles.